- API: https://api.weatherbit.io/v2.0/history/airquality?city=Hanoi&start_date=2024-02-08&end_date=2024-03-08&tz=local&key=API_KEY

In [ ]:
import requests
import pandas as pd
from datetime import datetime
from calendar import monthrange
import os
import time

✅ Đã import các thư viện cần thiết


In [ ]:
def get_weather_data(start_date, end_date, api_key):
    """
    Lấy dữ liệu chất lượng không khí từ Weatherbit API.
    
    Parameters:
    -----------
    start_date : str
        Ngày bắt đầu (định dạng: 'YYYY-MM-DD')
    end_date : str
        Ngày kết thúc (định dạng: 'YYYY-MM-DD')
    api_key : str
        API key của Weatherbit
        
    Returns:
    --------
    df : pandas DataFrame hoặc None
        DataFrame chứa dữ liệu hoặc None nếu có lỗi
    """
    url = "https://api.weatherbit.io/v2.0/history/airquality"
    params = {
        'city': 'Hanoi',
        'start_date': start_date,
        'end_date': end_date,
        'tz': 'local',
        'key': api_key
    }

    try:
        # Gọi API với timeout và xử lý SSL
        response = requests.get(url, params=params, verify=False, timeout=30)
        response.raise_for_status()  # Raise exception cho HTTP errors
        
        data = response.json()
        records = data.get('data', [])
        
        if records:
            df = pd.DataFrame(records)
            print(f"Lấy được {len(df)} bản ghi từ {start_date} đến {end_date}")
            return df
        else:
            print(f"⚠️  Không có dữ liệu cho khoảng {start_date} đến {end_date}")
            return None
            
    except requests.exceptions.Timeout:
        print(f"Timeout khi lấy dữ liệu từ {start_date} đến {end_date}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi lấy dữ liệu từ {start_date} đến {end_date}: {e}")
        return None
    except Exception as e:
        print(f"Lỗi không xác định: {e}")
        return None

In [ ]:
def save_data_to_csv(data, file_name):
    """
    Lưu dữ liệu vào file CSV.
    
    Parameters:
    -----------
    data : pandas DataFrame
        DataFrame chứa dữ liệu cần lưu
    file_name : str
        Tên file CSV
    """
    if data is not None and not data.empty:
        # Kiểm tra file đã tồn tại chưa
        file_exists = os.path.isfile(file_name)
        
        # Lưu dữ liệu (append nếu file đã tồn tại)
        data.to_csv(
            file_name, 
            index=False, 
            mode='a' if file_exists else 'w',
            header=not file_exists
        )
        
        print(f"Đã lưu {len(data)} bản ghi vào {file_name}")
        
        # Hiển thị tổng số bản ghi trong file
        if file_exists:
            total_records = len(pd.read_csv(file_name))
            print(f"Tổng số bản ghi trong file: {total_records}")
    else:
        print("Không có dữ liệu hợp lệ để lưu")

In [6]:
def get_months_between_dates(start_date, end_date):
    """
    Lấy danh sách các khoảng thời gian theo tháng giữa hai ngày.
    
    Parameters:
    -----------
    start_date : datetime
        Ngày bắt đầu
    end_date : datetime
        Ngày kết thúc
        
    Returns:
    --------
    months : list of tuples
        Danh sách các tuple (start_date, end_date) cho mỗi tháng
    """
    months = []
    start_year, start_month = start_date.year, start_date.month
    end_year, end_month = end_date.year, end_date.month

    while start_year < end_year or (start_year == end_year and start_month <= end_month):
        # Tạo ngày bắt đầu tháng
        month_start = datetime(start_year, start_month, 1)
        
        # Lấy ngày cuối tháng
        _, last_day = monthrange(start_year, start_month)
        month_end = datetime(start_year, start_month, last_day)
        
        # Đảm bảo không vượt quá end_date
        if month_end > end_date:
            month_end = end_date

        months.append((
            month_start.strftime('%Y-%m-%d'), 
            month_end.strftime('%Y-%m-%d')
        ))

        # Chuyển sang tháng tiếp theo
        if start_month == 12:
            start_month = 1
            start_year += 1
        else:
            start_month += 1

    return months

In [ ]:
# Cấu hình
API_KEY = 'bd78df23972d4c5787e72fd978e7c5cb'
OUTPUT_FILE = 'weather_data_hanoi.csv'

# Xác định khoảng thời gian
start_date = datetime(2023, 1, 1)
end_date = datetime.today()

print("="*60)
print("CẤU HÌNH LẤY DỮ LIỆU THỜI TIẾT")
print("="*60)
print(f"Từ ngày: {start_date.strftime('%Y-%m-%d')}")
print(f"Đến ngày: {end_date.strftime('%Y-%m-%d')}")
print(f"File output: {OUTPUT_FILE}")

# Lấy danh sách các tháng
months = get_months_between_dates(start_date, end_date)
print(f"\nTổng số tháng cần lấy dữ liệu: {len(months)}")
print(f"Danh sách các khoảng thời gian:")
for i, (month_start, month_end) in enumerate(months[:5], 1):
    print(f"   {i}. {month_start} → {month_end}")
if len(months) > 5:
    print(f"   ... và {len(months) - 5} tháng khác")
print("="*60)

CẤU HÌNH LẤY DỮ LIỆU THỜI TIẾT
📅 Từ ngày: 2023-01-01
📅 Đến ngày: 2025-11-10
📁 File output: weather_data_hanoi.csv

📊 Tổng số tháng cần lấy dữ liệu: 35
📋 Danh sách các khoảng thời gian:
   1. 2023-01-01 → 2023-01-31
   2. 2023-02-01 → 2023-02-28
   3. 2023-03-01 → 2023-03-31
   4. 2023-04-01 → 2023-04-30
   5. 2023-05-01 → 2023-05-31
   ... và 30 tháng khác


In [ ]:
print("\n" + "="*60)
print("BẮT ĐẦU LẤY DỮ LIỆU")
print("="*60 + "\n")

# Thống kê
total_months = len(months)
success_count = 0
failed_count = 0
total_records = 0

# Lấy dữ liệu cho từng tháng
for index, (month_start, month_end) in enumerate(months, 1):
    print(f"\n[{index}/{total_months}] 📡 Đang lấy dữ liệu: {month_start} → {month_end}")
    
    # Lấy dữ liệu từ API
    data = get_weather_data(month_start, month_end, API_KEY)
    
    # Lưu dữ liệu
    if data is not None:
        save_data_to_csv(data, OUTPUT_FILE)
        success_count += 1
        total_records += len(data)
    else:
        failed_count += 1
    
    # Delay để tránh rate limiting (nếu cần)
    if index < total_months:
        time.sleep(1)  # Chờ 1 giây trước khi request tiếp theo

# Tóm tắt kết quả
print("\n" + "="*60)
print("KẾT QUẢ LẤY DỮ LIỆU")
print("="*60)
print(f"Thành công: {success_count}/{total_months} tháng")
print(f"Thất bại: {failed_count}/{total_months} tháng")
print(f"Tổng số bản ghi: {total_records}")
print(f"File đã lưu: {OUTPUT_FILE}")
print("="*60)

# Đọc và hiển thị thông tin file CSV
if os.path.isfile(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"\nTHÔNG TIN FILE CSV:")
    print(f"   - Tổng số dòng: {len(df)}")
    print(f"   - Tổng số cột: {len(df.columns)}")
    print(f"   - Các cột: {list(df.columns[:10])}")
    if len(df.columns) > 10:
        print(f"              ...và {len(df.columns) - 10} cột khác")
    print(f"\n5 dòng đầu tiên:")
    print(df.head())


BẮT ĐẦU LẤY DỮ LIỆU


[1/35] 📡 Đang lấy dữ liệu: 2023-01-01 → 2023-01-31


c:\Users\63200744\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weatherbit.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Lấy được 721 bản ghi từ 2023-01-01 đến 2023-01-31
💾 Đã lưu 721 bản ghi vào weather_data_hanoi.csv

[2/35] 📡 Đang lấy dữ liệu: 2023-02-01 → 2023-02-28

[2/35] 📡 Đang lấy dữ liệu: 2023-02-01 → 2023-02-28


c:\Users\63200744\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weatherbit.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Lấy được 649 bản ghi từ 2023-02-01 đến 2023-02-28
💾 Đã lưu 649 bản ghi vào weather_data_hanoi.csv
📊 Tổng số bản ghi trong file: 1370

[3/35] 📡 Đang lấy dữ liệu: 2023-03-01 → 2023-03-31

[3/35] 📡 Đang lấy dữ liệu: 2023-03-01 → 2023-03-31


...

✅ Lấy được 697 bản ghi từ 2025-09-01 đến 2025-09-30
💾 Đã lưu 697 bản ghi vào weather_data_hanoi.csv
📊 Tổng số bản ghi trong file: 23337

[34/35] 📡 Đang lấy dữ liệu: 2025-10-01 → 2025-10-31

[34/35] 📡 Đang lấy dữ liệu: 2025-10-01 → 2025-10-31


c:\Users\63200744\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weatherbit.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Lấy được 721 bản ghi từ 2025-10-01 đến 2025-10-31
💾 Đã lưu 721 bản ghi vào weather_data_hanoi.csv
📊 Tổng số bản ghi trong file: 24058

[35/35] 📡 Đang lấy dữ liệu: 2025-11-01 → 2025-11-10

[35/35] 📡 Đang lấy dữ liệu: 2025-11-01 → 2025-11-10


c:\Users\63200744\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weatherbit.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Lấy được 217 bản ghi từ 2025-11-01 đến 2025-11-10
💾 Đã lưu 217 bản ghi vào weather_data_hanoi.csv
📊 Tổng số bản ghi trong file: 24275

KẾT QUẢ LẤY DỮ LIỆU
✅ Thành công: 35/35 tháng
❌ Thất bại: 0/35 tháng
📊 Tổng số bản ghi: 24275
📁 File đã lưu: weather_data_hanoi.csv

📈 THÔNG TIN FILE CSV:
   - Tổng số dòng: 24275
   - Tổng số cột: 11
   - Các cột: ['aqi', 'co', 'datetime', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'timestamp_local', 'timestamp_utc']
              ...và 1 cột khác

📊 5 dòng đầu tiên:
   aqi     co       datetime    no2    o3   pm10   pm25    so2  \
0  160  340.0  2023-01-30:17   47.3  46.0   78.8   63.0  110.0   
1  173  357.9  2023-01-30:16   50.7  53.0   91.3   73.0  121.0   
2  195  375.8  2023-01-30:15   54.0  60.0  111.3   89.0  132.0   
3  211  454.8  2023-01-30:14   77.3  57.7  126.3  101.0  135.3   
4  258  533.8  2023-01-30:13  100.7  55.3  171.3  137.0  138.7   

       timestamp_local        timestamp_utc          ts  
0  2023-01-31T00:00:00  2023-01-30T17:00:00 